# 01 — Exploratory Data Analysis

Look at the Lending Club data before doing anything else. The point of this notebook is to understand the shape of the problem, the target's distribution, where the missing data is, and how each feature actually relates to default.

Fill in the TODO cells, run them, and write your own observations in the markdown blocks. Don't skip the writing — that's the point.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('../data/loan.csv', low_memory=False)
print(f'{len(df):,} rows, {len(df.columns)} columns')

## Schema

What columns exist, what types, what's missing.

In [ ]:
# TODO: df.info(), df.dtypes.value_counts()
# How many numeric vs object vs other?


In [ ]:
# TODO: missing-value count per column, sorted descending
# missing = df.isna().sum().sort_values(ascending=False)
# missing[missing > 0]

**Observations**

_Where is missing data concentrated? Are those columns we'd drop anyway? Any features with so much missing they're not usable?_

## Target distribution

What are we predicting? How balanced?

In [ ]:
# TODO: df['loan_status'].value_counts(dropna=False)
# Statuses: Fully Paid, Charged Off, Current, In Grace Period, ...
# What fraction of all loans are settled (Fully Paid or Charged Off)?
# What's the charge-off rate among settled loans?

**Observations**

_How balanced is the binary target? Severe imbalance changes how you evaluate (use AUC and PR-AUC, not accuracy). Class weighting?_

## Filter to settled loans

Matching what train.py does — keep only loans with a known final outcome.

In [ ]:
settled = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
settled['target'] = (settled['loan_status'] == 'Charged Off').astype(int)
print(f'{len(settled):,} settled loans, default rate {settled["target"].mean():.1%}')

## Numeric feature distributions

Some columns are stored as strings (`13.5%`, `36 months`) and need parsing first. Then look at distributions — outliers, skew, anything weird.

In [ ]:
# Parse the text-encoded numerics
settled['int_rate_pct'] = settled['int_rate'].astype(str).str.rstrip('%').replace('nan', np.nan).astype(float)
settled['revol_util_pct'] = settled['revol_util'].astype(str).str.rstrip('%').replace('nan', np.nan).astype(float)
settled['term_months'] = settled['term'].astype(str).str.strip().str.split().str[0].replace('nan', np.nan).astype(float)

In [ ]:
FEATURES_NUMERIC = [
    'loan_amnt', 'int_rate_pct', 'installment', 'annual_inc', 'dti',
    'delinq_2yrs', 'fico_range_low', 'inq_last_6mths', 'open_acc',
    'pub_rec', 'revol_bal', 'revol_util_pct', 'total_acc',
]

# TODO: settled[FEATURES_NUMERIC].describe()
# Any features with extreme max values that look like data errors?
# (annual_inc has reported maxes in the millions — outliers? Or real?)

In [ ]:
# TODO: histograms for each
# settled[FEATURES_NUMERIC].hist(figsize=(15, 12), bins=40)

**Observations**

_What's heavily skewed (income, revol_bal)? Any bimodal distributions? Should anything be log-transformed before modeling? Any features where the histogram doesn't match what you'd expect?_

## Default rate by feature (bivariate)

The single most useful EDA plot for a classifier: bin the feature, compute the target rate per bin, plot. If the line slopes up (or down) monotonically, you've found signal.

In [ ]:
# Example for FICO
# settled['fico_bin'] = pd.qcut(settled['fico_range_low'], 10, duplicates='drop')
# settled.groupby('fico_bin')['target'].mean().plot(kind='bar')
# plt.title('Default rate by FICO decile')
# plt.ylabel('Default rate')
# plt.show()

# TODO: do the same for at least 5 other numeric features.
# Which show clean monotonic patterns? Which are flat (= no signal)?
# Any non-monotonic features that need binning rather than as-is?

**Observations**

_Rank the numeric features by how clearly they separate defaults from non-defaults. The strong ones are doing the heavy lifting; the weak ones might be droppable._

## Categorical features

Default rate by category.

In [ ]:
FEATURES_CATEGORICAL = ['grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'term']

# TODO: for each, show count + default rate per category
# for col in FEATURES_CATEGORICAL:
#     summary = settled.groupby(col)['target'].agg(['count', 'mean']).sort_values('mean')
#     print(f'\n=== {col} ===')
#     print(summary)

**Observations**

_How well does Lending Club's own grade rank-order default rates? (That's their internal model — your model needs to do at least as well.) Any state effects? Any purposes that are notably risky?_

## Correlations

Which numerics move together? Redundant pairs are pruning candidates.

In [ ]:
# TODO: correlation heatmap of numerics + target
# corr = settled[FEATURES_NUMERIC + ['target']].corr()
# sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f')

**Observations**

_Any features so correlated they're redundant? (loan_amnt and installment, for instance — installment is loan_amnt × rate × term.) What correlates strongest with the target?_

## Bottom line

_Based on this EDA, what would you change about train.py's feature list? Anything to add? Anything to drop? Any data-quality concerns to handle (outlier capping, log transforms)?_